# CAMTS-UNet: Cross-Attention Multi-Scale UNet
## Explainable Brain Tumor Segmentation and Classification on Radiologist-Annotated MRI
### NeuroLens — Department of Biomedical Engineering, Islamic University, Kushtia, Bangladesh

---
##  Notebook Structure
1. Environment Setup
2. Imports & Configuration
3. Dataset Description Diagram
4. Dataset Loading & Preprocessing
5. Augmentation & DataLoader
6. Methodology Pipeline Diagram
7. CAMTS-UNet Architecture
8. Loss Function
9. Metrics
10. Training Loop
11. Training Curves
12. Test Set Evaluation
13. Quantitative Analysis — Donut Charts
14. Qualitative Analysis — Input/GT/Pred/Overlay Grid
15. GradCAM Explainability
16. Pretrained Model Comparison (MHD Bar Chart)
17. Full Metrics Comparison Table
18. Single Image Inference


## 1.  Install Dependencies

In [ ]:
import importlib.util

required = ['torch', 'torchvision', 'cv2', 'albumentations', 'segmentation_models_pytorch']
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(f'Missing dependencies: {missing}. Install them before running this notebook.')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 3060


## 2. Imports & Configuration

In [ ]:
import os, cv2, time, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

class Config:
    DATA_DIR    = 'archive/kaggle_3m'
    IMAGE_SIZE  = 256
    BATCH_SIZE  = 8
    EPOCHS      = 50
    LR          = 1e-4
    DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED        = 42
    VAL_SPLIT   = 0.15
    TEST_SPLIT  = 0.15
    CHECKPOINT  = 'camts_unet_best.pth'
    MEAN        = (0.485, 0.456, 0.406)
    STD         = (0.229, 0.224, 0.225)

cfg = Config()
fallback_dirs = ['archive/kaggle_3m', 'archive/lgg-mri-segmentation/kaggle_3m']
for d in fallback_dirs:
    if Path(d).exists():
        cfg.DATA_DIR = d
        break
if not Path(cfg.DATA_DIR).exists():
    raise FileNotFoundError('Could not locate dataset under archive/.')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print(f'Device: {cfg.DEVICE}')


Device: cuda


## 3. Dataset Description Diagram

In [ ]:
fig = plt.figure(figsize=(20, 5))
fig.suptitle('Brain MRI Data Overview — LGG Segmentation Dataset', fontsize=14, fontweight='bold', y=1.02)

# 1. Patient split pie
ax1 = fig.add_subplot(141)
ax1.pie([59, 51], labels=['Tumor\n(59)', 'No Tumor\n(51)'],
        colors=['#E74C3C','#2ECC71'], autopct='%1.1f%%',
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Total Subjects\n(110 Patients)', fontweight='bold')

# 2. Imaging modalities
ax2 = fig.add_subplot(142)
modalities = ['FLAIR', 'T1', 'T1ce', 'T2']
counts     = [3929, 3929, 3929, 3929]
colors_mod = ['#3498DB','#9B59B6','#E67E22','#1ABC9C']
bars = ax2.bar(modalities, counts, color=colors_mod, edgecolor='white', linewidth=1.5)
ax2.set_title('Imaging Modalities', fontweight='bold')
ax2.set_ylabel('Slices')
for bar, c in zip(bars, counts):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20, str(c),
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. Pathological breakdown
ax3 = fig.add_subplot(143)
categories = ['No Tumor', 'Benign', 'Malignant']
cat_counts = [1571, 1180, 1178]
cat_colors = ['#95A5A6','#F39C12','#E74C3C']
bars2 = ax3.bar(categories, cat_counts, color=cat_colors, edgecolor='white', linewidth=1.5)
ax3.set_title('Pathological Images', fontweight='bold')
ax3.set_ylabel('Image Count')
for bar, c in zip(bars2, cat_counts):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, str(c),
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Dataset split
ax4 = fig.add_subplot(144)
splits = [70, 15, 15]
split_labels = ['Train\n70%', 'Val\n15%', 'Test\n15%']
split_colors = ['#3498DB','#E67E22','#2ECC71']
left = 0
for s, l, c in zip(splits, split_labels, split_colors):
    ax4.barh(0, s, left=left, color=c, edgecolor='white', linewidth=2, height=0.5)
    ax4.text(left + s/2, 0, l, ha='center', va='center',
             fontsize=10, fontweight='bold', color='white')
    left += s
ax4.set_xlim(0, 100)
ax4.set_ylim(-0.5, 0.5)
ax4.axis('off')
ax4.set_title('Dataset Split', fontweight='bold')

plt.tight_layout()
plt.savefig('dataset_description.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dataset_description.png')

Saved: dataset_description.png


## 4. Dataset Loading & Preprocessing

In [ ]:
def collect_pairs(data_dir):
    images, masks = [], []
    for patient_dir in sorted(Path(data_dir).iterdir()):
        if not patient_dir.is_dir(): continue
        for img_file in sorted(patient_dir.glob('*.tif')):
            if '_mask' not in img_file.name:
                mask_file = patient_dir / img_file.name.replace('.tif', '_mask.tif')
                if mask_file.exists():
                    images.append(str(img_file))
                    masks.append(str(mask_file))
    print(f'Total pairs: {len(images)}')
    return images, masks

all_images, all_masks = collect_pairs(cfg.DATA_DIR)

pos_imgs, pos_masks, neg_imgs, neg_masks = [], [], [], []
for img, mask in zip(all_images, all_masks):
    m = np.array(Image.open(mask).convert('L'))
    if m.max() > 0: pos_imgs.append(img); pos_masks.append(mask)
    else:           neg_imgs.append(img); neg_masks.append(mask)

n_neg = min(len(neg_imgs), len(pos_imgs))
idx   = np.random.choice(len(neg_imgs), n_neg, replace=False)
final_imgs  = pos_imgs + [neg_imgs[i] for i in idx]
final_masks = pos_masks + [neg_masks[i] for i in idx]
print(f'Positive: {len(pos_imgs)} | Negative (balanced): {n_neg} | Total: {len(final_imgs)}')

X_tr, X_tmp, y_tr, y_tmp = train_test_split(final_imgs, final_masks,
    test_size=cfg.VAL_SPLIT+cfg.TEST_SPLIT, random_state=cfg.SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp,
    test_size=0.5, random_state=cfg.SEED)
print(f'Train: {len(X_tr)} | Val: {len(X_val)} | Test: {len(X_te)}')

Total pairs: 3929


Positive: 1373 | Negative (balanced): 1373 | Total: 2746
Train: 1922 | Val: 412 | Test: 412


## 5. Augmentation & DataLoader

In [ ]:
train_transform = A.Compose([
    A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, p=0.4),
    A.ElasticTransform(alpha=120, sigma=6, p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=cfg.MEAN, std=cfg.STD),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
    A.Normalize(mean=cfg.MEAN, std=cfg.STD),
    ToTensorV2()
])

class BrainMRIDataset(Dataset):
    def __init__(self, images, masks, transform=None):
        self.images = images; self.masks = masks; self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(self.images[idx]), cv2.COLOR_BGR2RGB)
        mask = (np.array(Image.open(self.masks[idx]).convert('L')) > 0).astype(np.float32)
        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']; mask = aug['mask'].unsqueeze(0)
        return img, mask

train_ds = BrainMRIDataset(X_tr,  y_tr,  train_transform)
val_ds   = BrainMRIDataset(X_val, y_val, val_transform)
test_ds  = BrainMRIDataset(X_te,  y_te,  val_transform)
loader_workers = 0 if os.name == 'nt' else 2
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=loader_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=loader_workers)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=loader_workers)
print(f'Batches — Train:{len(train_loader)} Val:{len(val_loader)} Test:{len(test_loader)}')


Batches — Train:241 Val:52 Test:52


## 6. Methodology Pipeline Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
ax.set_xlim(0, 20); ax.set_ylim(0, 6); ax.axis('off')
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')

def draw_box(ax, x, y, w, h, text, color, fontsize=9):
    box = FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle='round,pad=0.1', facecolor=color,
        edgecolor='#555555', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True,
            multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', color='#333333',
                        lw=2, connectionstyle='arc3,rad=0'))

# Row 1 — Data stages
stage_color   = '#AED6F1'
proc_color    = '#A9DFBF'
model_color   = '#FAD7A0'
eval_color    = '#D7BDE2'

draw_box(ax, 2,   4.5, 2.8, 0.9, 'Data\nCollection',      stage_color)
draw_box(ax, 5.5, 4.5, 2.8, 0.9, 'Data\nPreprocessing',   proc_color)
draw_box(ax, 9,   4.5, 2.8, 0.9, 'Dataset\nSplit',        stage_color)
draw_arrow(ax, 3.4, 4.5, 4.1, 4.5)
draw_arrow(ax, 6.9, 4.5, 7.6, 4.5)

# Dataset split sub-boxes
draw_box(ax, 7.8, 3.2, 1.5, 0.6, 'Train 70%',  '#3498DB', fontsize=8)
draw_box(ax, 9.3, 3.2, 1.5, 0.6, 'Val 15%',    '#E67E22', fontsize=8)
draw_box(ax,10.8, 3.2, 1.5, 0.6, 'Test 15%',   '#2ECC71', fontsize=8)
draw_arrow(ax, 9, 4.05, 9, 3.55)

# Row 2 — Preprocessing steps
draw_box(ax, 2,   2.2, 2.8, 0.9, '3D to 2D\nSlice',       proc_color)
draw_box(ax, 5.5, 2.2, 2.8, 0.9, 'Image\nResizing 256×256', proc_color)
draw_box(ax, 9,   2.2, 2.8, 0.9, 'Normalization\n& Augmentation', proc_color)
draw_arrow(ax, 2, 4.05, 2, 2.65)
draw_arrow(ax, 3.4, 2.2, 4.1, 2.2)
draw_arrow(ax, 6.9, 2.2, 7.6, 2.2)

# Row 3 — Model & Evaluation
draw_box(ax, 12.5, 2.2, 3.0, 0.9, 'Proposed\nCAMTS-UNet',   model_color)
draw_box(ax, 16,   2.2, 2.8, 0.9, 'Model\nEvaluation',      eval_color)
draw_box(ax, 19,   2.2, 1.8, 0.9, 'Result\nAnalysis',       eval_color)
draw_arrow(ax, 10.4, 2.2, 11.0, 2.2)
draw_arrow(ax, 14.0, 2.2, 14.6, 2.2)
draw_arrow(ax, 17.4, 2.2, 18.1, 2.2)

ax.set_title('CAMTS-UNet: Methodology Pipeline', fontsize=14,
             fontweight='bold', pad=10)

# Legend
legend_elements = [
    mpatches.Patch(facecolor=stage_color,  label='Data Stages'),
    mpatches.Patch(facecolor=proc_color,   label='Processing'),
    mpatches.Patch(facecolor=model_color,  label='Model'),
    mpatches.Patch(facecolor=eval_color,   label='Evaluation'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('methodology_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: methodology_pipeline.png')

Saved: methodology_pipeline.png


## 7. CAMTS-UNet Architecture

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1, dilation=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = ConvBNReLU(ch, ch)
        self.conv2 = ConvBNReLU(ch, ch)
    def forward(self, x): return x + self.conv2(self.conv1(x))

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.res  = ResBlock(out_ch)
    def forward(self, x): return self.res(self.down(x))

class CrossAttentionGate(nn.Module):
    """
    Cross-Attention Gate: generates spatial attention map from
    gating signal (decoder) and skip connection (encoder).
    Suppresses irrelevant encoder features before concatenation.
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape != x1.shape:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=True)
        return x * self.psi(self.relu(g1 + x1))

class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling — Multi-Scale Feature Fusion"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1  = ConvBNReLU(in_ch, out_ch, kernel=1, padding=0)
        self.conv6  = ConvBNReLU(in_ch, out_ch, dilation=6,  padding=6)
        self.conv12 = ConvBNReLU(in_ch, out_ch, dilation=12, padding=12)
        self.conv18 = ConvBNReLU(in_ch, out_ch, dilation=18, padding=18)
        self.pool   = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.ReLU(inplace=True))
        self.project = ConvBNReLU(out_ch*5, out_ch, kernel=1, padding=0)
    def forward(self, x):
        size = x.shape[2:]
        p    = F.interpolate(self.pool(x), size=size, mode='bilinear', align_corners=True)
        return self.project(torch.cat([self.conv1(x),self.conv6(x),self.conv12(x),self.conv18(x),p], dim=1))

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(ConvBNReLU(out_ch+skip_ch, out_ch), ConvBNReLU(out_ch, out_ch))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.conv(torch.cat([x, skip], dim=1))

class CAMTSUNet(nn.Module):
    """
    CAMTS-UNet: Cross-Attention Multi-Scale UNet
    Encoder  : 4-scale ResBlock encoder [64,128,256,512]
    Bottleneck: ASPP multi-scale context
    Decoder  : 4 stages with Cross-Attention gated skip connections
    Output   : Binary segmentation mask
    """
    def __init__(self, in_ch=3, out_ch=1, features=[64,128,256,512]):
        super().__init__()
        self.stem       = nn.Sequential(ConvBNReLU(in_ch, features[0]), ResBlock(features[0]))
        self.enc1       = EncoderBlock(features[0], features[1])
        self.enc2       = EncoderBlock(features[1], features[2])
        self.enc3       = EncoderBlock(features[2], features[3])
        self.bottleneck = ASPP(features[3], features[3])
        self.ag3        = CrossAttentionGate(features[3], features[2], features[2]//2)
        self.ag2        = CrossAttentionGate(features[2], features[1], features[1]//2)
        self.ag1        = CrossAttentionGate(features[1], features[0], features[0]//2)
        self.dec3       = DecoderBlock(features[3], features[2], features[2])
        self.dec2       = DecoderBlock(features[2], features[1], features[1])
        self.dec1       = DecoderBlock(features[1], features[0], features[0])
        self.final_conv = nn.Conv2d(features[0], out_ch, kernel_size=1)
        self.ds3        = nn.Conv2d(features[2], out_ch, 1)
        self.ds2        = nn.Conv2d(features[1], out_ch, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.enc1(s0)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        b  = self.bottleneck(s3)
        d3 = self.dec3(b,  self.ag3(b,  s2))
        d2 = self.dec2(d3, self.ag2(d3, s1))
        d1 = self.dec1(d2, self.ag1(d2, s0))
        out = self.final_conv(d1)
        if self.training:
            ds3 = F.interpolate(self.ds3(d3), size=x.shape[2:], mode='bilinear', align_corners=True)
            ds2 = F.interpolate(self.ds2(d2), size=x.shape[2:], mode='bilinear', align_corners=True)
            return out, ds2, ds3
        return out

model = CAMTSUNet().to(cfg.DEVICE)
with torch.no_grad():
    model.eval()
    out = model(torch.randn(2,3,256,256).to(cfg.DEVICE))
print(f'Output shape: {out.shape}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Output shape: torch.Size([2, 1, 256, 256])
Trainable params: 19,884,073


## 8. Loss Function — Dice + BCE + Deep Supervision

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        pred   = torch.sigmoid(pred).view(-1)
        target = target.view(-1)
        inter  = (pred * target).sum()
        return 1 - (2.*inter + self.smooth) / (pred.sum() + target.sum() + self.smooth)

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, ds_weight=0.3):
        super().__init__()
        self.dice = DiceLoss()
        self.bce  = nn.BCEWithLogitsLoss()
        self.bce_w = bce_weight
        self.ds_w  = ds_weight
    def forward(self, outputs, target):
        if isinstance(outputs, tuple):
            main, ds2, ds3 = outputs
            main_loss = self.dice(main,target) + self.bce_w*self.bce(main,target)
            ds2_loss  = self.dice(ds2, target) + self.bce_w*self.bce(ds2, target)
            ds3_loss  = self.dice(ds3, target) + self.bce_w*self.bce(ds3, target)
            return main_loss + self.ds_w*(ds2_loss + ds3_loss)
        return self.dice(outputs,target) + self.bce_w*self.bce(outputs,target)

criterion = CombinedLoss()
print('Loss: Dice + BCE + Deep Supervision ready')

Loss: Dice + BCE + Deep Supervision ready


## 9. Metrics — Dice, IoU, Hausdorff Distance, Precision, Recall, F1

In [ ]:
def compute_metrics(pred_logits, target, threshold=0.5, smooth=1e-6):
    pred   = (torch.sigmoid(pred_logits) > threshold).float()
    pred_f = pred.view(-1)
    tgt_f  = target.view(-1)

    TP = (pred_f * tgt_f).sum().item()
    FP = (pred_f * (1 - tgt_f)).sum().item()
    FN = ((1 - pred_f) * tgt_f).sum().item()
    TN = ((1 - pred_f) * (1 - tgt_f)).sum().item()

    accuracy  = (TP + TN + smooth) / (TP + TN + FP + FN + smooth)
    precision = (TP + smooth) / (TP + FP + smooth)
    recall    = (TP + smooth) / (TP + FN + smooth)
    dice      = (2 * TP + smooth) / (2 * TP + FP + FN + smooth)
    iou       = (TP + smooth) / (TP + FP + FN + smooth)
    f1        = dice

    return {
        'accuracy': accuracy,
        'dice': dice,
        'iou': iou,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

def hausdorff_distance(pred_logits, target, threshold=0.5):
    pred_np = (torch.sigmoid(pred_logits) > threshold).cpu().numpy().squeeze()
    tgt_np  = target.cpu().numpy().squeeze()
    hds = []
    for p, t in zip(pred_np, tgt_np):
        if p.max()==0 and t.max()==0: hds.append(0.0); continue
        if p.max()==0 or  t.max()==0: hds.append(np.inf); continue
        d1  = distance_transform_edt(~p.astype(bool))
        d2  = distance_transform_edt(~t.astype(bool))
        hds.append(max(d1[t.astype(bool)].max(), d2[p.astype(bool)].max()))
    vals = [h for h in hds if h != np.inf]
    return np.nanmean(vals) if vals else 0.0

print('Metrics ready: Accuracy, Dice, IoU, Precision, Recall, F1, Hausdorff Distance')


Metrics ready: Accuracy, Dice, IoU, Precision, Recall, F1, Hausdorff Distance


## 10. Training Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-5)
history   = {'train_loss':[], 'val_loss':[], 'val_dice':[], 'val_iou':[]}
best_dice = 0.0

def train_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for imgs, masks in tqdm(loader, desc='Train', leave=False):
        imgs, masks = imgs.to(cfg.DEVICE), masks.to(cfg.DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def val_epoch(model, loader, criterion):
    model.eval(); total_loss = 0
    all_metrics = {'accuracy':0,'dice':0,'iou':0,'precision':0,'recall':0,'f1':0}
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Val', leave=False):
            imgs, masks = imgs.to(cfg.DEVICE), masks.to(cfg.DEVICE)
            preds = model(imgs)
            total_loss += criterion(preds, masks).item()
            m = compute_metrics(preds, masks)
            for k in all_metrics:
                all_metrics[k] += m[k]
    n = len(loader)
    return total_loss / n, {k: v / n for k, v in all_metrics.items()}

train_epochs = cfg.EPOCHS
if Path(cfg.CHECKPOINT).exists():
    model.load_state_dict(torch.load(cfg.CHECKPOINT, map_location=cfg.DEVICE))
    _, vm0 = val_epoch(model, val_loader, criterion)
    best_dice = vm0['dice']
    train_epochs = min(5, cfg.EPOCHS)
    print(f'Loaded checkpoint {cfg.CHECKPOINT} | Initial Val Dice: {best_dice:.4f}')
    print(f'Fine-tuning for {train_epochs} epoch(s) to refresh training curves.')

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=1e-6)

print('Starting training...')
for epoch in range(1, train_epochs + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vm = val_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_dice'].append(vm['dice'])
    history['val_iou'].append(vm['iou'])

    if vm['dice'] > best_dice:
        best_dice = vm['dice']
        torch.save(model.state_dict(), cfg.CHECKPOINT)
        print(f'  Saved best model (Dice:{best_dice:.4f})')

    if epoch % max(1, train_epochs // 5) == 0 or epoch == 1:
        print(f'Epoch[{epoch:3d}/{train_epochs}] '
              f'TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} '
              f'Dice:{vm["dice"]:.4f} IoU:{vm["iou"]:.4f} '
              f'Prec:{vm["precision"]:.4f} Rec:{vm["recall"]:.4f} '
              f'({time.time()-t0:.1f}s)')

print(f'Training complete. Best Dice: {best_dice:.4f}')


Val:   0%|          | 0/52 [00:00<?, ?it/s]

Val:   2%|▏         | 1/52 [00:01<01:32,  1.81s/it]

Val:   4%|▍         | 2/52 [00:01<00:41,  1.22it/s]

Val:   6%|▌         | 3/52 [00:02<00:24,  1.98it/s]

Val:   8%|▊         | 4/52 [00:02<00:17,  2.82it/s]

...

Val:  92%|█████████▏| 48/52 [00:06<00:00,  7.81it/s]

Val:  94%|█████████▍| 49/52 [00:06<00:00,  7.80it/s]

Val:  96%|█████████▌| 50/52 [00:06<00:00,  7.81it/s]

Val:  98%|█████████▊| 51/52 [00:06<00:00,  7.77it/s]

Epoch[  5/5] TrLoss:0.1728 VlLoss:0.1153 Dice:0.9056 IoU:0.8311 Prec:0.9013 Rec:0.9139 (102.5s)
Training complete. Best Dice: 0.9080


## 11. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if len(history['train_loss']) > 0:
    axes[0].plot(history['train_loss'], label='Train', color='tomato')
    axes[0].plot(history['val_loss'], label='Val', color='steelblue')
    axes[1].plot(history['val_dice'], color='green')
    axes[2].plot(history['val_iou'], color='purple')
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No training history available', ha='center', va='center', fontsize=11)

axes[0].set_title('Loss Curve'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Validation Dice Score'); axes[1].set_xlabel('Epoch')
axes[2].set_title('Validation IoU Score'); axes[2].set_xlabel('Epoch')
plt.suptitle('CAMTS-UNet Training Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')


Saved: training_curves.png


## 12. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(cfg.CHECKPOINT, map_location=cfg.DEVICE))
model.eval()

test_metrics = {'accuracy':0,'dice':0,'iou':0,'precision':0,'recall':0,'f1':0}
test_hd = 0
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Testing'):
        imgs, masks = imgs.to(cfg.DEVICE), masks.to(cfg.DEVICE)
        preds = model(imgs)
        m = compute_metrics(preds, masks)
        for k in test_metrics: test_metrics[k] += m[k]
        test_hd += hausdorff_distance(preds, masks)

n = len(test_loader)
final_metrics = {k: v/n*100 for k,v in test_metrics.items()}
final_hd      = test_hd / n

print('\n' + '='*55)
print('          CAMTS-UNet — TEST RESULTS')
print('='*55)
for k,v in final_metrics.items():
    print(f'  {k.capitalize():<20}: {v:.2f}%')
print(f'  {"Hausdorff Dist (MHD)":<20}: {final_hd:.4f}')
print('='*55)


Testing:   0%|          | 0/52 [00:00<?, ?it/s]

Testing:   2%|▏         | 1/52 [00:00<00:06,  7.32it/s]

Testing:   4%|▍         | 2/52 [00:00<00:06,  7.24it/s]

Testing:   6%|▌         | 3/52 [00:00<00:06,  7.12it/s]

Testing:   8%|▊         | 4/52 [00:00<00:06,  7.18it/s]

...

Testing:  94%|█████████▍| 49/52 [00:06<00:00,  7.10it/s]

Testing:  96%|█████████▌| 50/52 [00:06<00:00,  7.20it/s]

Testing:  98%|█████████▊| 51/52 [00:07<00:00,  7.20it/s]

Testing: 100%|██████████| 52/52 [00:07<00:00,  7.22it/s]


          CAMTS-UNet — TEST RESULTS
  Accuracy            : 99.71%
  Dice                : 88.46%
  Iou                 : 80.10%
  Precision           : 88.54%
  Recall              : 89.54%
  F1                  : 88.46%
  Hausdorff Dist (MHD): 5.4867


## 13. Quantitative Analysis — Donut Charts (Section 7)

In [ ]:
def get_color(value):
    if value >= 90: return '#27AE60'
    elif value >= 80: return '#F39C12'
    else: return '#E74C3C'

metric_names = ['Dice Score', 'IoU Score', 'Precision', 'Recall', 'F1 Score']
metric_keys  = ['dice', 'iou', 'precision', 'recall', 'f1']
metric_values = [final_metrics[k] for k in metric_keys]

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle('CAMTS-UNet: Quantitative Analysis', fontsize=15, fontweight='bold', y=1.02)

for ax, name, value in zip(axes, metric_names, metric_values):
    color = get_color(value)
    ax.pie([value, 100-value],
           colors=[color, '#EAECEE'],
           startangle=90,
           wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
           counterclock=False)
    ax.text(0, 0.08, f'{value:.2f}%', ha='center', va='center',
            fontsize=13, fontweight='bold', color=color)
    ax.text(0, -0.25, name, ha='center', va='center',
            fontsize=10, fontweight='bold', color='#2C3E50')
    circle = plt.Circle((0,0), 0.55, color='white')
    ax.add_patch(circle)

plt.tight_layout()
plt.savefig('quantitative_donut_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: quantitative_donut_charts.png')

Saved: quantitative_donut_charts.png

## 14. Qualitative Analysis — Input / GT / Pred / Overlay

In [ ]:
def denormalize(tensor, mean=cfg.MEAN, std=cfg.STD):
    img = tensor.permute(1,2,0).cpu().numpy()
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)

def make_overlay(img_np, pred_mask, gt_mask):
    """Color overlay: TP=Green, FP=Red, FN=Blue"""
    overlay = img_np.copy()
    pred_b  = pred_mask.astype(bool)
    gt_b    = gt_mask.astype(bool)
    TP = pred_b & gt_b
    FP = pred_b & ~gt_b
    FN = ~pred_b & gt_b
    alpha = 0.45
    overlay[TP] = overlay[TP]*(1-alpha) + np.array([0,1,0])*alpha  # Green
    overlay[FP] = overlay[FP]*(1-alpha) + np.array([1,0,0])*alpha  # Red
    overlay[FN] = overlay[FN]*(1-alpha) + np.array([0,0,1])*alpha  # Blue
    return np.clip(overlay, 0, 1)

# Pick 2 samples: largest tumor & smallest tumor
tumor_sizes = [(i, np.array(Image.open(y_te[i]).convert('L')).sum()) for i in range(len(y_te))]
tumor_positive = [(i,s) for i,s in tumor_sizes if s > 0]
tumor_positive.sort(key=lambda x: x[1])
idx_small = tumor_positive[0][0]
idx_large = tumor_positive[-1][0]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
col_titles = ['Input MRI', 'Ground Truth', 'Predicted Mask', 'Overlay (TP=G, FP=R, FN=B)']
row_labels  = ['Large Tumor', 'Small Tumor']

for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=11, fontweight='bold')

model.eval()
for row, idx in enumerate([idx_large, idx_small]):
    img_t, mask_t = test_ds[idx]
    img_t_batch = img_t.unsqueeze(0).to(cfg.DEVICE)
    with torch.no_grad():
        pred_logit = model(img_t_batch)
    pred_mask = (torch.sigmoid(pred_logit).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    gt_mask   = mask_t.squeeze().numpy()
    img_np    = denormalize(img_t)
    overlay   = make_overlay(img_np, pred_mask, gt_mask)

    axes[row][0].imshow(img_np)
    axes[row][1].imshow(gt_mask,   cmap='gray')
    axes[row][2].imshow(pred_mask, cmap='gray')
    axes[row][3].imshow(overlay)
    axes[row][0].set_ylabel(row_labels[row], fontsize=11, fontweight='bold')
    for ax in axes[row]: ax.axis('off')

plt.suptitle('CAMTS-UNet: Qualitative Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('qualitative_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: qualitative_analysis.png')

Saved: qualitative_analysis.png


## 15. GradCAM — Explainability

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        self._hooks      = [
            target_layer.register_forward_hook(lambda m,i,o: setattr(self,'activations',o.detach())),
            target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,'gradients',go[0].detach()))
        ]
    def generate(self, x):
        self.model.eval()
        x   = x.requires_grad_(True)
        out = self.model(x)
        self.model.zero_grad()
        out.mean().backward()
        w   = self.gradients.mean(dim=[2,3], keepdim=True)
        cam = F.relu((w * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=True)
        cam = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    def remove(self):
        for h in self._hooks: h.remove()

gradcam = GradCAM(model, model.bottleneck.project)
sample_imgs, sample_masks = next(iter(test_loader))
sample_imgs  = sample_imgs[:4].to(cfg.DEVICE)
sample_masks = sample_masks[:4]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ['MRI Image', 'Ground Truth', 'Predicted Mask', 'GradCAM (Explainability)']
for col, t in enumerate(col_titles):
    axes[0][col].set_title(t, fontsize=11, fontweight='bold')

for i in range(4):
    img_np   = denormalize(sample_imgs[i])
    gt_np    = sample_masks[i].squeeze().numpy()
    with torch.no_grad():
        pred = (torch.sigmoid(model(sample_imgs[i:i+1])).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    cam = gradcam.generate(sample_imgs[i:i+1])
    axes[i][0].imshow(img_np);           axes[i][0].axis('off')
    axes[i][1].imshow(gt_np, cmap='gray'); axes[i][1].axis('off')
    axes[i][2].imshow(pred,  cmap='gray'); axes[i][2].axis('off')
    axes[i][3].imshow(img_np)
    axes[i][3].imshow(cam, alpha=0.55, cmap='jet')
    axes[i][3].axis('off')

gradcam.remove()
plt.suptitle('CAMTS-UNet: GradCAM Explainability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('predictions_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: predictions_gradcam.png')

Saved: predictions_gradcam.png


## 16. Pretrained Model Comparison — MHD Bar Chart

In [ ]:
baselines = {
    'UNet-VGG16'      : smp.Unet('vgg16',          encoder_weights=None, in_channels=3, classes=1),
    'UNet-ResNet34'   : smp.Unet('resnet34',        encoder_weights=None, in_channels=3, classes=1),
    'UNet-ResNet50'   : smp.Unet('resnet50',        encoder_weights=None, in_channels=3, classes=1),
    'UNet-EffNetB0'   : smp.Unet('efficientnet-b0', encoder_weights=None, in_channels=3, classes=1),
    'UNet-MobileNetV2': smp.Unet('mobilenet_v2',    encoder_weights=None, in_channels=3, classes=1),
}

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

def evaluate_model(model, loader, device):
    model.eval().to(device)
    metrics = {'accuracy':0,'dice':0,'iou':0,'precision':0,'recall':0,'f1':0}
    hd_total = 0
    t0 = time.time()
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Eval', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            m = compute_metrics(preds, masks)
            for k in metrics:
                metrics[k] += m[k]
            hd_total += hausdorff_distance(preds, masks)
    n = len(loader)
    return {k: v/n*100 for k, v in metrics.items()}, hd_total/n, time.time()-t0

# Short baseline training for comparison
BASELINE_EPOCHS = 1
comparison_results = {}

for name, bmodel in baselines.items():
    print('')
    print(f'Training {name}...')
    bmodel = bmodel.to(cfg.DEVICE)
    opt = torch.optim.AdamW(bmodel.parameters(), lr=cfg.LR, weight_decay=1e-5)
    crit = CombinedLoss()
    epoch_time = 0

    for ep in range(BASELINE_EPOCHS):
        t0 = time.time()
        bmodel.train()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(cfg.DEVICE), masks.to(cfg.DEVICE)
            opt.zero_grad()
            loss = crit(bmodel(imgs), masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(bmodel.parameters(), 1.0)
            opt.step()
        epoch_time += time.time()-t0

    m, hd, _ = evaluate_model(bmodel, test_loader, cfg.DEVICE)
    comparison_results[name] = {
        **m,
        'mhd': hd,
        'params': count_params(bmodel),
        'epoch_time': epoch_time / BASELINE_EPOCHS
    }
    print(f'{name} -> Dice:{m["dice"]:.2f}% MHD:{hd:.4f} Params:{count_params(bmodel):.1f}M')

# Add CAMTS-UNet results
camts_m, camts_hd, _ = evaluate_model(model, test_loader, cfg.DEVICE)
comparison_results['CAMTS-UNet (Ours)'] = {
    **camts_m,
    'mhd': camts_hd,
    'params': count_params(model),
    'epoch_time': 0
}

# MHD bar chart
model_names = list(comparison_results.keys())
mhd_values = [comparison_results[m]['mhd'] for m in model_names]
bar_colors = ['#5DADE2'] * len(model_names)
camts_idx = model_names.index('CAMTS-UNet (Ours)')
bar_colors[camts_idx] = '#1A5276'

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(model_names, mhd_values, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
ax.axhline(y=mhd_values[camts_idx], color='#E74C3C', linestyle='--', linewidth=1.5,
           label=f'CAMTS-UNet MHD = {mhd_values[camts_idx]:.3f}')

best_idx = int(np.argmin(mhd_values))
for i, (bar, val) in enumerate(zip(bars, mhd_values)):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    if i == best_idx:
        bar.set_edgecolor('#E74C3C')
        bar.set_linewidth(3)
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.012, '* Best', ha='center',
                fontsize=11, color='#E74C3C', fontweight='bold')

ax.set_title('MHD Comparison (Lower is Better)', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean Hausdorff Distance (MHD)')
ax.legend(fontsize=10)
ax.set_facecolor('#FDFEFE')
plt.tight_layout()
plt.savefig('mhd_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: mhd_comparison.png')



Training UNet-VGG16...


Eval:   0%|          | 0/52 [00:00<?, ?it/s]

Eval:   4%|▍         | 2/52 [00:00<00:03, 12.69it/s]

Eval:   8%|▊         | 4/52 [00:00<00:03, 12.91it/s]

Eval:  12%|█▏        | 6/52 [00:00<00:03, 13.28it/s]

...

Eval:  92%|█████████▏| 48/52 [00:06<00:00,  7.08it/s]

Eval:  94%|█████████▍| 49/52 [00:06<00:00,  7.12it/s]

Eval:  96%|█████████▌| 50/52 [00:06<00:00,  7.21it/s]

Eval:  98%|█████████▊| 51/52 [00:07<00:00,  7.19it/s]

Saved: mhd_comparison.png


## 17. Full Metrics Comparison Table

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

col_labels = ['Model', 'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)']
table_data = []
for name, res in comparison_results.items():
    table_data.append([
        name.replace('\n', ' '),
        f'{res["accuracy"]:.2f}',
        f'{res["precision"]:.2f}',
        f'{res["recall"]:.2f}',
        f'{res["f1"]:.2f}'
    ])

table = ax.table(cellText=table_data, colLabels=col_labels,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

for j in range(len(col_labels)):
    table[0, j].set_facecolor('#1A5276')
    table[0, j].set_text_props(color='white', fontweight='bold')

camts_row = None
for idx, row in enumerate(table_data, start=1):
    if 'CAMTS-UNet' in row[0]:
        camts_row = idx
        break

if camts_row is not None:
    for j in range(len(col_labels)):
        table[camts_row, j].set_facecolor('#D5F5E3')
        table[camts_row, j].set_text_props(fontweight='bold')

for i in range(1, len(table_data) + 1):
    if camts_row is not None and i == camts_row:
        continue
    color = '#EBF5FB' if i % 2 == 0 else 'white'
    for j in range(len(col_labels)):
        table[i, j].set_facecolor(color)

ax.set_title('Model Comparison: Accuracy, Precision, Recall, F1', fontsize=13,
             fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('accuracy_precision_recall_f1_table.png', dpi=150, bbox_inches='tight')
plt.savefig('model_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_precision_recall_f1_table.png')
print('Saved: model_comparison_table.png')


Saved: accuracy_precision_recall_f1_table.png
Saved: model_comparison_table.png


## 18. Single Image Inference

In [ ]:
def predict_single(image_path, model, transform, device):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    aug = transform(image=img, mask=np.zeros(img.shape[:2], dtype=np.float32))
    inp = aug['image'].unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out      = model(inp)
        prob_map = torch.sigmoid(out).squeeze().cpu().numpy()
        pred_mask = (prob_map > 0.5).astype(np.uint8)
    return pred_mask, prob_map, img

sample_path = X_te[0]
pred_mask, prob_map, orig_img = predict_single(sample_path, model, val_transform, cfg.DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_img);               axes[0].set_title('Input MRI',       fontweight='bold'); axes[0].axis('off')
axes[1].imshow(prob_map, cmap='hot');   axes[1].set_title('Probability Map',  fontweight='bold'); axes[1].axis('off')
axes[2].imshow(pred_mask, cmap='gray'); axes[2].set_title('Segmented Tumor',  fontweight='bold'); axes[2].axis('off')
plt.suptitle('CAMTS-UNet: Single Image Inference', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('single_inference.png', dpi=150, bbox_inches='tight')
plt.show()

has_tumor = pred_mask.max() > 0
print(f'Tumor detected : {"YES ✅" if has_tumor else "NO ❌"}')
print(f'Tumor area     : {pred_mask.sum()} pixels ({pred_mask.mean()*100:.2f}% of image)')

Tumor detected : NO ❌
Tumor area     : 0 pixels (0.00% of image)


---
## Complete Output Files Summary

| # | File | Description |
|---|------|-------------|
| 1 | `dataset_description.png`       | Patient split, modalities, pathology, dataset split |
| 2 | `methodology_pipeline.png`      | Full preprocessing → model → evaluation flow |
| 3 | `training_curves.png`           | Loss, Dice, IoU over epochs |
| 4 | `quantitative_donut_charts.png` | 5 circular metric charts (Dice/IoU/Prec/Rec/F1) |
| 5 | `qualitative_analysis.png`      | Input/GT/Pred/Overlay grid (large & small tumor) |
| 6 | `predictions_gradcam.png`       | GradCAM explainability 4×4 grid |
| 7 | `mhd_comparison.png`            | MHD bar chart vs 5 pretrained baselines |
| 8 | `model_comparison_table.png`    | Full metrics table all models |
| 9 | `single_inference.png`          | Single MRI inference output |
|10 | `camts_unet_best.pth`           | Best model checkpoint |

## ⚡ Tips to Improve Results
- Use **mixed precision** (`torch.cuda.amp`) for 2× faster training
- Add **Test Time Augmentation (TTA)** for better predictions
- Try **Lovász loss** for sharper boundary segmentation
- Use **pretrained ResNet encoder** in CAMTS-UNet stem for transfer learning
- Ensemble multiple checkpoints for final submission

In [ ]:
# Advanced evaluation diagnostics: per-case metrics and sampled pixel scores
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve, auc

per_case_metrics = []
sampled_probs = []
sampled_targets = []
sample_budget = 2_000_000
rng = np.random.default_rng(cfg.SEED)

model.eval()
with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Advanced Eval', leave=False):
        imgs, masks = imgs.to(cfg.DEVICE), masks.to(cfg.DEVICE)
        preds = model(imgs)
        probs = torch.sigmoid(preds)

        # Per-case metrics
        for i in range(preds.shape[0]):
            p_i = preds[i:i+1]
            m_i = masks[i:i+1]
            met = compute_metrics(p_i, m_i)

            pred_bin = (torch.sigmoid(p_i) > 0.5).float()
            p_flat = pred_bin.view(-1)
            t_flat = m_i.view(-1)
            tp = float((p_flat * t_flat).sum().item())
            fp = float((p_flat * (1 - t_flat)).sum().item())
            fn = float(((1 - p_flat) * t_flat).sum().item())
            tn = float(((1 - p_flat) * (1 - t_flat)).sum().item())

            per_case_metrics.append({
                **met,
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'tn': tn,
                'gt_pixels': float(m_i.sum().item()),
                'pred_pixels': float(pred_bin.sum().item())
            })

        # Sample pixel probabilities for ROC/PR/confusion charts
        if sample_budget > 0:
            probs_np = probs.detach().cpu().numpy().reshape(-1)
            t_np = masks.detach().cpu().numpy().reshape(-1).astype(np.uint8)

            k = min(sample_budget, probs_np.size)
            if k < probs_np.size:
                idx = rng.choice(probs_np.size, size=k, replace=False)
                sampled_probs.append(probs_np[idx])
                sampled_targets.append(t_np[idx])
            else:
                sampled_probs.append(probs_np)
                sampled_targets.append(t_np)
            sample_budget -= k

sampled_probs = np.concatenate(sampled_probs) if sampled_probs else np.array([], dtype=np.float32)
sampled_targets = np.concatenate(sampled_targets) if sampled_targets else np.array([], dtype=np.uint8)

print(f'Per-case samples: {len(per_case_metrics)}')
print(f'Sampled pixels for curves: {len(sampled_probs)}')


Advanced Eval:   0%|          | 0/52 [00:00<?, ?it/s]

Advanced Eval:   2%|▏         | 1/52 [00:00<00:06,  7.51it/s]

Advanced Eval:   4%|▍         | 2/52 [00:00<00:06,  7.37it/s]

Advanced Eval:   6%|▌         | 3/52 [00:00<00:06,  7.31it/s]

Advanced Eval:   8%|▊         | 4/52 [00:00<00:06,  7.15it/s]

...

Advanced Eval:  92%|█████████▏| 48/52 [00:06<00:00,  7.48it/s]

Advanced Eval:  94%|█████████▍| 49/52 [00:06<00:00,  7.50it/s]

Advanced Eval:  96%|█████████▌| 50/52 [00:06<00:00,  7.50it/s]

Advanced Eval:  98%|█████████▊| 51/52 [00:06<00:00,  7.50it/s]

Per-case samples: 412
Sampled pixels for curves: 2000000


In [ ]:
# Additional evaluation diagrams
metrics_for_dist = ['accuracy', 'dice', 'iou', 'precision', 'recall', 'f1']
metric_labels = ['Accuracy', 'Dice', 'IoU', 'Precision', 'Recall', 'F1']

# 1) Distribution of per-case metrics
fig, ax = plt.subplots(figsize=(12, 6))
box_data = [[m[k] * 100 for m in per_case_metrics] for k in metrics_for_dist]
box = ax.boxplot(box_data, labels=metric_labels, patch_artist=True, showmeans=True)
colors = ['#5DADE2', '#58D68D', '#F5B041', '#AF7AC5', '#EC7063', '#45B39D']
for patch, c in zip(box['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.65)
ax.set_ylabel('Score (%)')
ax.set_title('Per-Case Metric Distributions on Test Set', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('test_case_metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: test_case_metric_distributions.png')

# 2) Dice vs ground-truth tumor size
x = np.array([m['gt_pixels'] for m in per_case_metrics], dtype=np.float64)
y = np.array([m['dice'] * 100 for m in per_case_metrics], dtype=np.float64)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, s=22, alpha=0.6, color='#2E86C1', edgecolors='none')
if len(x) > 1:
    p = np.polyfit(np.log10(x + 1), y, deg=1)
    xx = np.linspace(x.min(), x.max(), 300)
    yy = p[0] * np.log10(xx + 1) + p[1]
    ax.plot(xx, yy, color='#C0392B', linewidth=2, label='Trend')
ax.set_xscale('log')
ax.set_xlabel('Ground-Truth Tumor Pixels (log scale)')
ax.set_ylabel('Dice Score (%)')
ax.set_title('Dice vs Tumor Size', fontweight='bold')
ax.grid(alpha=0.3)
if len(x) > 1:
    ax.legend()
plt.tight_layout()
plt.savefig('dice_vs_tumor_size.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dice_vs_tumor_size.png')

# 3) Pixel-level confusion matrix (from sampled pixels)
if sampled_probs.size > 0 and sampled_targets.size > 0:
    pred_bin = (sampled_probs >= 0.5).astype(np.uint8)
    cm = confusion_matrix(sampled_targets, pred_bin, labels=[0, 1])
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    ax.set_title('Pixel-Level Confusion Matrix (Normalized)', fontweight='bold')
    for i in range(2):
        for j in range(2):
            txt = f'{cm_norm[i, j]:.3f}' + '\n' + f'({cm[i, j]})'
            ax.text(j, i, txt, ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig('pixel_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: pixel_confusion_matrix.png')

# 4) ROC + Precision-Recall curves
if sampled_probs.size > 0 and sampled_targets.size > 0 and sampled_targets.min() != sampled_targets.max():
    fpr, tpr, _ = roc_curve(sampled_targets, sampled_probs)
    roc_auc = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(sampled_targets, sampled_probs)
    pr_auc = auc(rec, prec)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(fpr, tpr, color='#1F77B4', linewidth=2, label=f'AUC = {roc_auc:.4f}')
    axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve', fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(rec, prec, color='#D35400', linewidth=2, label=f'AUC = {pr_auc:.4f}')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve', fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: roc_pr_curves.png')

# 5) Core metric comparison bar chart across models
core_metrics = ['accuracy', 'precision', 'recall', 'f1']
core_labels = ['Accuracy', 'Precision', 'Recall', 'F1']
models = list(comparison_results.keys())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
for ax, mk, ml in zip(axes, core_metrics, core_labels):
    vals = [comparison_results[m][mk] for m in models]
    bars = ax.bar(models, vals, color=['#7FB3D5' if 'CAMTS' not in m else '#1F618D' for m in models])
    best_idx = int(np.argmax(vals))
    bars[best_idx].set_edgecolor('#C0392B')
    bars[best_idx].set_linewidth(2)
    ax.set_title(f'{ml} Comparison', fontweight='bold')
    ax.set_ylabel('Score (%)')
    ax.tick_params(axis='x', rotation=25)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f'{v:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison Across Core Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('core_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: core_metrics_comparison.png')


Saved: test_case_metric_distributions.png


Saved: dice_vs_tumor_size.png
Saved: pixel_confusion_matrix.png


Saved: roc_pr_curves.png


Saved: core_metrics_comparison.png


In [ ]:
# Extra evaluation diagrams (3): threshold sensitivity, calibration, and hardest-case error profile
from sklearn.calibration import calibration_curve

if sampled_probs.size > 0 and sampled_targets.size > 0:
    # 1) Threshold sensitivity curves
    thresholds = np.linspace(0.05, 0.95, 19)
    t_acc, t_prec, t_rec, t_f1, t_iou = [], [], [], [], []

    y_true = sampled_targets.astype(np.uint8)
    for th in thresholds:
        y_pred = (sampled_probs >= th).astype(np.uint8)

        tp = float(((y_pred == 1) & (y_true == 1)).sum())
        tn = float(((y_pred == 0) & (y_true == 0)).sum())
        fp = float(((y_pred == 1) & (y_true == 0)).sum())
        fn = float(((y_pred == 0) & (y_true == 1)).sum())

        smooth = 1e-6
        acc = (tp + tn + smooth) / (tp + tn + fp + fn + smooth)
        prec = (tp + smooth) / (tp + fp + smooth)
        rec = (tp + smooth) / (tp + fn + smooth)
        f1 = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou = (tp + smooth) / (tp + fp + fn + smooth)

        t_acc.append(acc * 100)
        t_prec.append(prec * 100)
        t_rec.append(rec * 100)
        t_f1.append(f1 * 100)
        t_iou.append(iou * 100)

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(thresholds, t_acc, label='Accuracy', linewidth=2)
    ax.plot(thresholds, t_prec, label='Precision', linewidth=2)
    ax.plot(thresholds, t_rec, label='Recall', linewidth=2)
    ax.plot(thresholds, t_f1, label='F1', linewidth=2)
    ax.plot(thresholds, t_iou, label='IoU', linewidth=2)

    best_idx = int(np.argmax(t_f1))
    best_th = thresholds[best_idx]
    ax.axvline(best_th, color='#C0392B', linestyle='--', linewidth=1.5, label=f'Best F1 threshold = {best_th:.2f}')

    ax.set_xlabel('Decision Threshold')
    ax.set_ylabel('Score (%)')
    ax.set_title('Threshold Sensitivity of Pixel-Level Metrics', fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.savefig('threshold_sensitivity_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: threshold_sensitivity_curves.png')

    # 2) Reliability diagram (calibration)
    frac_pos, mean_pred = calibration_curve(y_true, sampled_probs, n_bins=12, strategy='quantile')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    axes[0].plot(mean_pred, frac_pos, marker='o', linewidth=2, color='#1F77B4')
    axes[0].set_xlabel('Mean Predicted Probability')
    axes[0].set_ylabel('Observed Positive Rate')
    axes[0].set_title('Reliability Diagram', fontweight='bold')
    axes[0].grid(alpha=0.3)

    axes[1].hist(sampled_probs, bins=30, color='#7FB3D5', edgecolor='white')
    axes[1].set_xlabel('Predicted Probability')
    axes[1].set_ylabel('Pixel Count')
    axes[1].set_title('Probability Distribution', fontweight='bold')
    axes[1].grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig('reliability_diagram.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: reliability_diagram.png')

# 3) Hardest-case error breakdown
if len(per_case_metrics) > 0:
    # rank by lowest F1 / Dice
    ranked = sorted(per_case_metrics, key=lambda d: d['f1'])[:10]
    labels = [f'C{i+1}' for i in range(len(ranked))]
    fp_vals = [r['fp'] for r in ranked]
    fn_vals = [r['fn'] for r in ranked]
    tp_vals = [r['tp'] for r in ranked]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(labels, fn_vals, label='FN', color='#F1948A')
    ax.bar(labels, fp_vals, bottom=fn_vals, label='FP', color='#F8C471')
    ax.bar(labels, tp_vals, bottom=np.array(fn_vals) + np.array(fp_vals), label='TP', color='#82E0AA')

    for i, r in enumerate(ranked):
        ax.text(i, fn_vals[i] + fp_vals[i] + tp_vals[i] + max(tp_vals) * 0.01,
                f'F1={r["f1"]*100:.1f}%', ha='center', va='bottom', fontsize=8)

    ax.set_xlabel('Hardest Cases (lowest F1)')
    ax.set_ylabel('Pixel Count')
    ax.set_title('Error Composition of Hardest Test Cases', fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('hardest_cases_error_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: hardest_cases_error_breakdown.png')


Saved: threshold_sensitivity_curves.png


Saved: reliability_diagram.png
Saved: hardest_cases_error_breakdown.png
